In [1]:
import torch
import os
import requests
import tarfile
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, BlipForConditionalGeneration
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive') #liaison avec mon google drive

img_folder_path = '/content/drive/MyDrive/102flowers/jpg' #lien vers la base de données qui est dans mon google drive

#vérification
if os.path.exists(img_folder_path):
    print(f"Dossier trouvé ! Nombre d'images : {len(os.listdir(img_folder_path))}")
else:
    print("Chemin introuvable")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dossier trouvé ! Nombre d'images : 8189


In [3]:
!cp -r "/content/drive/MyDrive/102flowers/jpg" "/content/jpg_local" #copie le dossier dans le stockage direct de colab, askip c'est plus rapide
img_folder_path = "/content/jpg_local"

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu" #utilise le GPU a la place du CPU si c'est possible
print(f"Utilisation de : {device}")

#classe pour gérer la dataset. Fait avec le LLM Gemini
class OxfordFlowersDataset(Dataset):
    def __init__(self, img_dir):
        self.img_dir = img_dir
        self.img_names = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        image = Image.open(img_path).convert('RGB')
        return image, self.img_names[idx]

#chargement d'un modèle préexistant qui utilise les transformers. Plus récent que la méthode citée dans l'article et plus rapide à utiliser. Modèles proposées par le LLM
#la normalisation et le passage en .pth se font directement dedans donc pas besoin de le faire avant
processor = AutoProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

#méthode de l'article pour attribuer un attribut couleur à une image
def get_voted_color(all_descriptions):
    target_colors = ["red", "blue", "pink", "yellow", "purple", "white", "orange", "violet", "green"]
    found_colors = []

    for desc in all_descriptions:
        for color in target_colors:
            if color in desc.lower(): # si une des couleurs de target_colors est trouvé dans la phrase, alors on le note dans found_colors
                found_colors.append(color)

    if not found_colors: #si on trouve aucune couleur dans la phrase on met blanc par défaut (c'est un choix personnel, il n'y a pas de détail dans l'article sur ce cas)
        return "white"

    counts = Counter(found_colors) #instance de la classe Counter pour compter facilement les couleurs. proposé par le LLM
    # Règle : Seuil de 3 apparitions
    top_color, top_count = counts.most_common(1)[0] #donne la couleur la plus mentionné et le nombre de fois où elle l'est
    return top_color if top_count >= 3 else "white" #si la couleur est mentionnée + de 3x, alors on considère que ca sera l'attribut "couleur" de l'image. Si c'est moins alors c'est forcé en "white". C'est un choix personnel vu que c'est pas précisé comme faire dans l'article

#Toute la suite a été donnée par le LLM
dataset = OxfordFlowersDataset(img_folder_path)

BATCH_SIZE = 8

def collate_fn(batch):
    # batch est une liste de tuples (image, nom)
    images = [item[0] for item in batch]
    names = [item[1] for item in batch]
    return images, names

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

results = []

print(f"Début du traitement avec {device}...")
with torch.no_grad():
    for images, names in tqdm(dataloader):
        # 1. Préparation du batch pour le GPU
        inputs = processor(images=images, text=["a photo of a"] * len(images), return_tensors="pt", padding=True).to(device)

        # 2. Génération massive
        outputs = model.generate(
            **inputs,
            num_return_sequences=10, #10 phrases par images comme ca a été fait dans l'article
            do_sample=True, #permet de pas tout le temps donner la même phrase
            top_k=50, #paramètre pour gérer les variations qu'une phrase de description peut avoir je crois
            temperature=0.7,
            max_new_tokens=20 #taille max de la phrase en token
        )

        # 3. Analyse des résultats
        # Les outputs sont groupés par 10 (ex: index 0-9 pour la 1ère image, 10-19 pour la 2ème)
        for i in range(len(names)):
            start_idx = i * 10
            end_idx = (i + 1) * 10
            batch_descriptions = [processor.decode(o, skip_special_tokens=True) for o in outputs[start_idx:end_idx]]

            final_color = get_voted_color(batch_descriptions)
            results.append(f"{names[i]} {final_color}")

#sauvegarde du fichier des attributs dans mon drive
output_file = "/content/drive/MyDrive/102flowers/oxford102_colors_2.txt"
with open(output_file, "w") as f:
    f.write("\n".join(results))

print(f"\nTerminé ! Fichier sauvegardé sous : {output_file}")

Utilisation de : cuda
Début du traitement avec cuda...


100%|██████████| 1024/1024 [23:28<00:00,  1.38s/it]


Terminé ! Fichier sauvegardé sous : /content/drive/MyDrive/102flowers/oxford102_colors_2.txt
